# Random Forest and Bagging

This notebook compares a Decision Tree, Random Forest, and Bagging Classifiers on the Titanic dataset. It also demonstrates out-of-bag (OOB) evaluation. The preprocessing workflow uses a train/test split before fitting imputers and encoders.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

## 1. Load and prepare the dataset

The Titanic dataset is loaded from Seaborn. The split happens before preprocessing, so imputation statistics and category encodings are learned from the training set only.

In [ ]:
titanic = sns.load_dataset("titanic")

features = ["pclass", "sex", "fare", "embarked", "age"]
target = "survived"
X = titanic[features]
y = titanic[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

numeric_features = ["pclass", "fare", "age"]
categorical_features = ["sex", "embarked"]

numeric_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])
categorical_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_preprocessor, numeric_features),
    ("categorical", categorical_preprocessor, categorical_features)
])

## 2. Decision Tree baseline

A single Decision Tree gives us a baseline to compare with ensemble approaches. Training and test accuracy help reveal possible overfitting.

In [ ]:
decision_tree = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(max_depth=4, random_state=42))
])

decision_tree.fit(X_train, y_train)
y_pred_tree = decision_tree.predict(X_test)
y_train_pred_tree = decision_tree.predict(X_train)

print("Training accuracy:", accuracy_score(y_train, y_train_pred_tree))
print("Test accuracy:", accuracy_score(y_test, y_pred_tree))
print("\nClassification report:\n", classification_report(y_test, y_pred_tree))

## 3. Visualize the Decision Tree

Only the first three levels are displayed to keep the diagram readable.

In [ ]:
fitted_preprocessor = decision_tree.named_steps["preprocessor"]
fitted_tree = decision_tree.named_steps["classifier"]
feature_names = fitted_preprocessor.get_feature_names_out()

plt.figure(figsize=(20, 10))
plot_tree(
    fitted_tree,
    feature_names=feature_names,
    class_names=["Died", "Survived"],
    filled=True,
    rounded=True,
    max_depth=3
)
plt.title("Decision Tree (first three levels)")
plt.tight_layout()
plt.show()

## 4. Random Forest and out-of-bag (OOB) evaluation

Random Forest combines predictions from many randomized trees. The OOB score uses training observations left out of each tree's bootstrap sample. It is a useful internal estimate, but the separate test set remains important.

In [ ]:
random_forest = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=501,
        max_depth=4,
        oob_score=True,
        random_state=42,
        n_jobs=-1
    ))
])

random_forest.fit(X_train, y_train)
y_pred_rf = random_forest.predict(X_test)
rf_estimator = random_forest.named_steps["classifier"]

print("OOB score:", rf_estimator.oob_score_)
print("Test accuracy:", accuracy_score(y_test, y_pred_rf))
print("\nClassification report:\n", classification_report(y_test, y_pred_rf))

## 5. Bagging Classifier with a Decision Tree

Bagging trains base estimators on bootstrap samples and aggregates their predictions.

In [ ]:
bagging_tree = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=42),
        n_estimators=201,
        random_state=42,
        n_jobs=-1
    ))
])

bagging_tree.fit(X_train, y_train)
y_pred_bagging_tree = bagging_tree.predict(X_test)
print("Decision Tree Bagging accuracy:", accuracy_score(y_test, y_pred_bagging_tree))

## 6. Bagging Classifier with Logistic Regression

In [ ]:
bagging_logistic = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", BaggingClassifier(
        estimator=LogisticRegression(max_iter=1000),
        n_estimators=201,
        random_state=42,
        n_jobs=-1
    ))
])

bagging_logistic.fit(X_train, y_train)
y_pred_bagging_logistic = bagging_logistic.predict(X_test)
print("Logistic Regression Bagging accuracy:", accuracy_score(y_test, y_pred_bagging_logistic))

## 7. Compare test-set accuracy

In [ ]:
results = pd.DataFrame({
    "Model": ["Decision Tree", "Random Forest", "Bagging + Decision Tree", "Bagging + Logistic Regression"],
    "Test accuracy": [
        accuracy_score(y_test, decision_tree.predict(X_test)),
        accuracy_score(y_test, random_forest.predict(X_test)),
        accuracy_score(y_test, bagging_tree.predict(X_test)),
        accuracy_score(y_test, bagging_logistic.predict(X_test))
    ]
}).sort_values("Test accuracy", ascending=False)
results

## 8. Confusion matrix

In [ ]:
ConfusionMatrixDisplay.from_estimator(random_forest, X_test, y_test)
plt.title("Random Forest Confusion Matrix")
plt.tight_layout()
plt.show()

## Key takeaways and future extensions

- Preprocessing is fitted on training data through a pipeline to reduce leakage.
- One-hot encoding is used for nominal categorical features.
- OOB score complements test evaluation for the Random Forest model.
- Planned follow-ups: Bagging Regressor, Boosting, and deeper comparison using precision, recall, F1-score, and feature importance.